# Working with streams

A stream is a set of values with an optional index that places each value on a
shared timeline (a timestamp or sequence number). When streams tick on different
clocks, the operators respect that index rather than row position. The model
itself is described in [Streams, values, and alignment](../multistream.md) and the
[`Stream` reference](../functions_streams/Stream.md).

This notebook covers the operators that reshape streams: `combine_latest` aligns
streams on their index, `merge` and `split` interleave and separate them, and
`dropna` and `filter` drop events. The last three change the length of the stream.
All of them are causal, deciding only from events seen so far, and they compose.
Each returns values first, as `(values, index)`.

In [ ]:
import numpy as np
import pandas as pd
from screamer import combine_latest, merge, split, dropna
from screamer import filter as keep   # 'filter' shadows the builtin, so alias it

# two feeds on one timeline, ticking at different times
a_idx = np.array([1, 3, 5, 7, 9], dtype=np.int64)
a_val = np.array([10., 11., 12., 11., 13.])
b_idx = np.array([2, 4, 6, 8], dtype=np.int64)
b_val = np.array([5., 6., 5.5, 6.5])

## Combining streams on different clocks

Stream a ticks at odd indices, b at even, so they never share a timestamp.
`combine_latest` emits a row on every distinct index and forward-fills each
stream's latest value across the gaps (an as-of join). Watch a value repeat down
the rows where only the other stream ticked. With the default `emit="when_all"`,
output begins only once every stream has ticked at least once, so there are no
cold-start NaNs.

The aligned `(T, 2)` values feed straight into any function, for example
`RollingCorr(10)(aligned)`; inside a `Dag`, the shorter
`RollingCorr(10)(combine_latest(a, b))` form works too.

In [ ]:
aligned, idx = combine_latest(a_val, b_val, index=[a_idx, b_idx])
spread = aligned[:, 0] - aligned[:, 1]

pd.DataFrame({
    "a (carried)": aligned[:, 0],
    "b (carried)": aligned[:, 1],
    "spread": spread,
}, index=idx).rename_axis("index")

### Emit modes: `when_all` vs `on_any`

`emit="on_any"` starts at the very first event, filling streams not yet seen with
NaN until their first value arrives. `when_all` (used above) is the same result
without those leading NaN rows. Use `on_any` when you want the earliest possible
output and can drop the warm-up NaNs downstream, an idiom shown at the end.

In [ ]:
aligned_any, idx_any = combine_latest(a_val, b_val, index=[a_idx, b_idx], emit="on_any")
print(f"on_any: {len(idx_any)} rows, when_all: {len(idx)} rows")

pd.DataFrame({
    "a (carried)": aligned_any[:, 0],
    "b (carried)": aligned_any[:, 1],
}, index=idx_any).rename_axis("index")

## Interleaving and separating: `merge` and `split`

`merge` interleaves the streams into one index-sorted sequence, tagging each event
with the stream it came from. It does not align or forward-fill; it just orders
every event on one timeline. `split` is its inverse: given the tagged stream, it
routes the events back into per-source `(values, index)` pairs.

In [ ]:
values, sources, m_idx = merge(a_val, b_val, index=[a_idx, b_idx])

pd.DataFrame({
    "value": values,
    "source": np.where(sources == 0, "a", "b"),
}, index=m_idx).rename_axis("index")

In [ ]:
(a_back, a_back_idx), (b_back, b_back_idx) = split(values, sources, index=m_idx)

print("recovered a:", a_back, "at", a_back_idx)
print("recovered b:", b_back, "at", b_back_idx)
print("round trip matches the originals:",
      np.array_equal(a_back, a_val) and np.array_equal(b_back, b_val))

## Dropping events: `dropna`

`dropna` removes events whose value is NaN, giving a shorter, gap-free stream. For
a 2-D aligned stream, `how="any"` (the default) drops a row if any column is NaN,
while `how="all"` drops it only if every column is NaN.

In [ ]:
readings_idx = np.array([1, 2, 3, 4, 5], dtype=np.int64)
readings = np.array([1.0, np.nan, 3.0, -4.0, 5.0])

clean, clean_idx = dropna(readings, index=readings_idx)
print("input  :", readings, "at", readings_idx)
print("cleaned:", clean, "at", clean_idx, "(index 2 removed)")

# 2-D: how='any' vs how='all'
grid_idx = np.array([1, 2, 3], dtype=np.int64)
grid = np.array([[1.0, np.nan],
                 [np.nan, np.nan],
                 [3.0, 4.0]])
_, keep_any = dropna(grid, index=grid_idx, how="any")
_, keep_all = dropna(grid, index=grid_idx, how="all")

pd.DataFrame({
    "col 0": grid[:, 0],
    "col 1": grid[:, 1],
    "kept (how=any)": np.isin(grid_idx, keep_any),
    "kept (how=all)": np.isin(grid_idx, keep_all),
}, index=grid_idx).rename_axis("index")

## Keeping events: `filter`

`filter` keeps only the events for which a Python predicate returns true. The
predicate receives a scalar for a 1-D stream and a 1-D array for a 2-D stream. It
is the flexible escape hatch; for plain NaN removal prefer `dropna`, and for
threshold cuts on large arrays a NumPy mask is faster.

In [ ]:
kept, kept_idx = keep(readings, lambda v: v > 0, index=readings_idx)
print("kept:", kept, "at", kept_idx)

pd.DataFrame({
    "value": readings,
    "kept (v > 0)": [bool(v > 0) for v in readings],   # NaN > 0 is False, so NaN is dropped
}, index=readings_idx).rename_axis("index")

## Composing them

The operators chain. A common idiom removes the cold-start NaN that
`emit="on_any"` produces: align with `on_any` to start as early as possible, then
`dropna` to drop the warm-up rows, giving a clean aligned stream ready for any
function.

In [ ]:
raw, raw_idx = combine_latest(a_val, b_val, index=[a_idx, b_idx], emit="on_any")
clean, clean_idx = dropna(raw, index=raw_idx)

print("on_any rows :", len(raw_idx), "(the first has a NaN)")
print("after dropna:", len(clean_idx), "clean aligned rows")

pd.DataFrame({
    "a": raw[:, 0],
    "b": raw[:, 1],
    "kept (dropna)": ~np.isnan(raw).any(axis=1),
}, index=raw_idx).rename_axis("index")